# dynamic material cycle modeling: Estimating the material content of the global vehicle fleet with a stock-driven model
*ODYM example by Stefan Pauliuk, adapted for flodym, changed from inflow-driven to stock-driven model - prepared for the 2026 Bath summer school on dynamic MFA in Python*

<img src="pictures/Bath_Cover.png" width="850" alt="2026 Bath dMFA summer school">

This example shows a fully-fledged MFA system, designed to estimate the material composition of a multi-regional passenger vehicle fleet by 2060.

The research questions asked are: How big is the material stock embodied in the global passenger vehicle fleet, and when will this material become available for recycling?

To answer these questions a dynamic material flow analysis of the global passenger vehicle fleet and the waste management industries is performed.

The dynamic MFA model has the following indices:

* t: time (1993-2060)
* c: age-cohort (1993-2060) (internal only)
* r: region (40 countries)
* T: drive technology (gasoline and battery electric passenger vehicles)
* p: process (vehicle markets, use phase, waste management industries, scrap markets)
* m: engineering materials (25)

The system definition of the model is given in the figure below. The time frame is 1993-2060. The figure also shows the aspects of the different system variables. The total registration of vehicles, for example, is broken down into individual materials, whereas the flow of deregistered vehicles is broken down into regions, age-cohorts, and materials.

<img src="pictures/Vehicle_Fleet_MFA_scope_1_2_3.png" width="850" height="290" alt="MFA system">

The model equations are as given in the lecture slide deck (Bath_dMFA_Python_SummerSchool_2026.pdf), software 4.

The remaining system variables are calculated by mass balance.

## 1. Load flodym and other useful packages

In [ ]:
from os.path import join

import numpy as np
import pandas as pd
import plotly.express as px

from flodym import (
    DimensionDefinition,
    Dimension,
    DimensionSet,
    ParameterDefinition,
    Parameter,
    FlowDefinition,
    StockDefinition,
    MFASystem,
    MFADefinition,
    DataReader,
    StockDrivenDSM,
)
from flodym.lifetime_models import NormalLifetime

## 2. Define the data requirements, flows, stocks and MFA system equations

We define the dimensions that are relevant for our system and the model parameters, processes, stocks and flows. We further define a class with our system equations in the compute method.

In [ ]:
dimension_definitions = [
    DimensionDefinition(letter="t", name="time", dtype=int),
    DimensionDefinition(letter="r", name="region", dtype=str),
    DimensionDefinition(letter="m", name="material", dtype=str),
    DimensionDefinition(letter="T", name="drive technology", dtype=str),
]

process_names = ["sysenv", "primary material production", "manufacturing", "use", "waste mgt"]

parameter_definitions = [
    ParameterDefinition(name="vehicle lifetime by technology", dim_letters=("r", "T")),
    ParameterDefinition(name="vehicle material content by technology", dim_letters=("m", "T")),
    ParameterDefinition(name="vehicle historic future stock by technology", dim_letters=("t", "r", "T")),
    ParameterDefinition(name="EoL recovery rate", dim_letters=("m",)),
]

flow_definitions = [
    # Tbd.

stock_definitions = [
    StockDefinition(
        name="in use",
        process="use",
        dim_letters=("t", "r", "T"),
        subclass=StockDrivenDSM,
        lifetime_model_class=NormalLifetime,
        time_letter="t",
    )
]

mfa_definition = MFADefinition(
    dimensions=dimension_definitions,
    processes=process_names,
    flows=flow_definitions,
    stocks=stock_definitions,
    parameters=parameter_definitions,
)

In [ ]:
class VehicleMFA(MFASystem):
    """We just need to define the compute method with our system equations,
    as all the other things we need are inherited from the MFASystem class."""

    def compute(self):
        self.compute_stock()
        self.compute_flows()

    def compute_stock(self):
        # tbd.

    def compute_flows(self):
        # tbd.

## 3. Define our data reader
Now that we have defined the MFA system and know what data we need, we can load the data.
To do the data loading, we define a DataReader class. Such a class can be reused with different datasets of the same format by passing attributes, e.g. the directory where the data is stored, in the init function. In this example, we will also use this data reader in a following step.

In [ ]:
class CustomDataReader(DataReader):
    """The methods `read_dimensions` and `read_parameters` are already defined in the parent
    DataReader class, and loop over the methods `read_dimension` and `read_parameter_values`
    that we specify for our usecase here.

    This data reader is complete and needs no modification for this workbook to run.    
    """

    def __init__(self, data_directory, years):
        self.data_directory = data_directory
        self.years = years

    def read_dimension(self, dimension_definition: DimensionDefinition) -> Dimension:
        if (dim_name := dimension_definition.name) == "region":
            data = pd.read_excel(
                join(self.data_directory, "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_historic_future_stock_by_technology.xlsx"), "Data"
            )
            other_data = pd.read_excel(
                join(self.data_directory, "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_lifetime_by_technology.xlsx"), "Data"
            )
            data = (set(data[dim_name].unique())).intersection(set(other_data[dim_name].unique()))
            data = list(data)
            data.sort()
        elif dimension_definition.name == "drive technology":
            data = pd.read_excel(
                join(self.data_directory, "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_lifetime_by_technology.xlsx"), "Data"
            )
            data.columns = [x.lower() for x in data.columns]
            data = list(data["drive technology"].unique())
            data.sort()  
        elif dimension_definition.name == "material":
            data = pd.read_excel(
                join(self.data_directory, "flodym_Bath_dMFA_Global_Vehicle_Fleet_vehicle_material_content_by_technology.xlsx"), "Data"
            )
            data.columns = [x.lower() for x in data.columns]
            data = list(data["material"].unique())
            data.sort()           
        elif dimension_definition.name == "time":
            data = self.years
        return Dimension(
            name=dimension_definition.name,
            letter=dimension_definition.letter,
            dtype=dimension_definition.dtype,
            items=data,
        )

    def read_parameter_values(self, parameter_name: str, dims: DimensionSet) -> Parameter:
        data = pd.read_excel(
            join(self.data_directory, f"flodym_Bath_dMFA_Global_Vehicle_Fleet_{parameter_name.replace(' ', '_')}.xlsx"), "Data"
        )
        data = data.fillna(0)
        if "r" in dims:  # remove unwanted regions
            data = data[data["region"].isin(dims["r"].items)]

        if parameter_name == "vehicle lifetime by technology":
            data.columns = [x.lower() for x in data.columns]
            # remove unncessary columns
            data = data[[dim.name for dim in dims] + ["value"]]
        
        if parameter_name == "vehicle material content by technology":
            data.columns = [x.lower() for x in data.columns]
            # remove unncessary columns
            data = data[[dim.name for dim in dims] + ["value"]]

        if parameter_name == "EoL recovery rate":
            data.columns = [x.lower() for x in data.columns]
            # remove unncessary columns
            data = data[[dim.name for dim in dims] + ["value"]]            
            
        return Parameter.from_df(dims=dims, df=data)

## 4. Put everything together
We make an instance of our `CustomDataReader`, read in the data and use it to create an instance of our `VehicleMFA` class. Then we can run the calculations and check the results

In [ ]:
data_reader = CustomDataReader(data_directory="input_data", years=list(range(1993, 2061)))

vehicle_mfa = VehicleMFA.from_data_reader(
    definition=mfa_definition,
    data_reader=data_reader,
)
vehicle_mfa.compute()  

## 5. Answer research questions
Plot material stocks and flows:
- the final consumption of materials (flow into use phase)
- the total recycled material
- the recycled content or the share of primary production in final material demand

In [ ]:
vehicle_mfa.flows["manufacturing => use"]

In [ ]:
vehicle_mfa.flows["primary material production => manufacturing"]